In [ ]:
import os
import rawpy
from PIL import Image
import piexif
from tqdm import tqdm
def extract_gps_from_exif(exif_dict):
    """
    Extract GPS IFD and Image DateTime tag from EXIF dict.
    Returns a minimal EXIF dict with these tags only.
    """
    gps_ifd = exif_dict.get("GPS", {})

    zeroth_ifd = {}
    # Tag 0x0132 is DateTime in 0th IFD
    datetime_tag = 0x0132
    if datetime_tag in exif_dict.get("0th", {}):
        zeroth_ifd[datetime_tag] = exif_dict["0th"][datetime_tag]

    return {
        "0th": zeroth_ifd,
        "GPS": gps_ifd
    }

input_dir = "E:\Masterthesis_FIRO\\Mai\\raw"
output_dir = "E:\\Masterthesis_FIRO\\Mai\\jpg"
os.makedirs(output_dir, exist_ok=True)

for filename in tqdm(sorted(os.listdir(input_dir))):
    if not filename.lower().endswith(".dng"):
        continue
    
    filename = filename.replace("._","")
    out_name = filename.replace(".dng", ".jpg").replace(".DNG", ".jpg")
    jpgs = os.listdir(output_dir)
    if out_name in jpgs:
        continue
    dng_path = input_dir +"/"+filename
    with rawpy.imread(dng_path) as raw:
        rgb = raw.postprocess(no_auto_bright=True, use_camera_wb=True, output_bps=8)

    img = Image.fromarray(rgb)

    # Load full EXIF from DNG
    exif_dict = piexif.load(dng_path)

    # Extract only GPS tags
    gps_only_exif = extract_gps_from_exif(exif_dict)

    # Dump minimal GPS EXIF bytes
    exif_bytes = piexif.dump(gps_only_exif)

    # Save as JPEG with GPS EXIF only
    out_path = os.path.join(output_dir, out_name)
    img.save(out_path, "jpeg", exif=exif_bytes, quality=95)

print("GPS-only EXIF transfer complete.")

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\Masterthesis_FIRO\\Mai\\raw'

In [12]:
import os
import datetime
import matplotlib.pyplot as plt
import exifread
import requests
# Directory with DNG files
folder_path = './chateau_jpeg'
time_gap_threshold = 200  # seconds — threshold to split flights

# === DATA COLLECTION ===
images_data = []
# Lists to store timestamps and altitudes
time_seconds = []
altitudes = []
def get_elevation_open(lat, lon):
    url = f"https://api.open-elevation.com/api/v1/lookup?locations={lat},{lon}"
    response = requests.get(url)
    if response.status_code == 200:
        result = response.json()
        elevation = result['results'][0]['elevation']
        return elevation
    return None
def convert_gps_to_decimal(gps_tag):
    """
    Converts an exifread IfdTag GPS coordinate to decimal degrees.
    """
    # gps_tag.values is usually a list of Ratio objects
    degrees = gps_tag.values[0]
    minutes = gps_tag.values[1]
    seconds = gps_tag.values[2]

    def ratio_to_float(ratio):
        return float(ratio.num) / float(ratio.den)

    deg = ratio_to_float(degrees)
    min = ratio_to_float(minutes)
    sec = ratio_to_float(seconds)

    decimal = deg + (min / 60.0) + (sec / 3600.0)
    return decimal

In [13]:
for filename in sorted(os.listdir(folder_path)):
    if filename.lower().endswith('.jpg'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'rb') as f:
            tags = exifread.process_file(f, details=False)
            time_tag = tags.get('Image DateTime')
            if not time_tag:
                continue
            time_str = str(time_tag).split()[1]  
            h, m, s = map(int, time_str.split(':'))
            total_seconds = h * 3600 + m * 60 + s

            alt_tag = tags.get('GPS GPSAltitude')
            if not alt_tag:
                continue
            if '/' in str(alt_tag):
                num, denom = map(float, str(alt_tag).split('/'))
                altitude = num / denom
            else:
                altitude = float(str(alt_tag))
            lat_tag = tags.get("GPS GPSLatitude")
            if not lat_tag:
                continue
            """latitude = convert_gps_to_decimal(lat_tag)
            long_tag = tags.get("GPS GPSLongitude")
            longitude = convert_gps_to_decimal(long_tag)
            elevation = get_elevation_open(latitude, longitude)
            print(f"Elevation: {elevation} meters")"""

            images_data.append((total_seconds, altitude))


In [14]:
# === SPLIT INTO FLIGHTS ===
flights = []
current_flight = []

for i, (t, alt) in enumerate(images_data):
    if i == 0 or (t - images_data[i - 1][0] <= time_gap_threshold):
        current_flight.append((t, alt))
    else:
        flights.append(current_flight)
        current_flight = [(t, alt)]
if current_flight:
    flights.append(current_flight)

In [12]:
# === PLOT EACH FLIGHT ===
for i, flight in enumerate(flights, 1):
    times = [t for t, _ in flight]
    new_times = []
    for timen in times:
        timen2 = timen - times[0]
        new_times.append(timen2)
    alts = [a for _, a in flight]
    alts = [a - alts[0] for a in alts]
    #mean_alt = sum(alts) / len(alts)
    #rel_alts = [a - mean_alt for a in alts]

    plt.figure(figsize=(8, 5))
    plt.plot(new_times, alts, marker='o')
    plt.title(f"Flight {i}: Relative Altitude Change")
    plt.xlabel("Time (seconds since midnight)")
    plt.ylabel("Altitude (meters)")
    plt.grid(True)
    plt.tight_layout()
    #plt.show()

NameError: name 'flights' is not defined

In [69]:
from PIL import Image
import numpy as np
def convert2to1(img1, img2, indir, outdir):
    numb = img1.split("-")[1]
    image1 = Image.open(os.path.join(indir, img1)).convert("L")
    image2 = Image.open(os.path.join(indir, img2)).convert("L")
    image1_array = np.array(image1)
    image2_array = np.array(image2)
    newg = 150
    image2_array[image2_array != 255] = 0
    image1_array[image1_array != 255] = 0
    image2_array[image2_array == 255] = newg
    merged = np.maximum(image1_array, image2_array)
    merged_im = Image.fromarray(merged, mode="L")
    merged_im.save(os.path.join(outdir,f"mask_{numb}.png"))

indir ="E:\Masterthesis_FIRO\Mai\masks"
outdir="C:/Users/dmz-user/Desktop/mask_gen/thesis/images/masks"
img1list = []
img2list = []
for filename in sorted(os.listdir(indir)):
    if filename.lower().endswith(".png"):
        if "-mit" in filename:
            img2list.append(filename)
        else:
            img1list.append(filename)
for i in range(len(img1list)):
    convert2to1(img1list[i], img2list[i], indir, outdir)